# Improved RAG System for Question Answering
## Key Improvements:
1. **Query Expansion**: Reformulate queries to improve retrieval
2. **Hybrid Retrieval**: Combine multiple retrieval strategies
3. **Reranking**: Reorder retrieved documents for better relevance
4. **Optimized Prompts**: Better instruction design for answer extraction
5. **Answer Post-processing**: Clean and normalize answers
6. **Parameter Tuning**: Optimize BM25 mu and number of documents retrieved

## Installation and Setup

In [ ]:
# Install required packages
!pip install pyserini==0.36.0
!pip install torch torchvision torchaudio
!pip install transformers
!pip install sentence-transformers
!pip install pandas numpy

In [ ]:
# Hugging Face Authentication
hugging_face_token = "YOUR_HF_TOKEN_HERE"

from huggingface_hub import login
login(hugging_face_token)

## Load Index and Data

In [ ]:
from pyserini.search import SimpleSearcher
from pyserini.index.lucene import IndexReader
import pandas as pd
import json
import re
import string
from collections import Counter
import numpy as np

# Load Pyserini index
searcher = SimpleSearcher.from_prebuilt_index('wikipedia-kilt-doc')
index_reader = IndexReader.from_prebuilt_index('wikipedia-kilt-doc')

print("Index stats:", index_reader.stats())

In [ ]:
# Load data files
# IMPORTANT: Use converters parameter to properly parse the answers column as JSON
train_path = "/content/drive/MyDrive/q-a-rag-assignment-3-reichman-uni/train.csv"
test_path = "/content/drive/MyDrive/q-a-rag-assignment-3-reichman-uni/test.csv"

# Load training data with JSON converter for answers column
df_train = pd.read_csv(train_path, converters={"answers": json.loads})
# Load test data (no answers column)
df_test = pd.read_csv(test_path)

print(f"Train rows: {len(df_train)}")
print(f"Test rows: {len(df_test)}")
print("\nSample training data:")
print(df_train.head())

## Load LLM

In [ ]:
import transformers
import torch

model_id = "meta-llama/Llama-3.2-1B-Instruct"

pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto",
)

terminators = [
    pipeline.tokenizer.eos_token_id,
    pipeline.tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

## Improved Retrieval Functions

In [ ]:
# Query preprocessing and expansion
def preprocess_query(query):
    """
    Clean and normalize query text
    """
    # Remove extra whitespace
    query = ' '.join(query.split())
    
    # Fix common query patterns
    query = query.replace('what is the name of', 'name of')
    query = query.replace('what does', 'meaning of')
    query = query.replace('where is', 'location of')
    query = query.replace('who is', '')
    query = query.replace('when did', 'date')
    
    return query.strip()

def expand_query(query):
    """
    Generate query variations to improve retrieval
    """
    queries = [query]
    
    # Remove question words for keyword search
    question_words = ['what', 'where', 'when', 'who', 'which', 'how', 'why', 'is', 'are', 'does', 'do', 'did']
    tokens = query.lower().split()
    keyword_query = ' '.join([t for t in tokens if t not in question_words])
    if keyword_query and keyword_query != query:
        queries.append(keyword_query)
    
    return queries

In [ ]:
def get_context_improved(query, k=10, mu=1000):
    """
    Improved retrieval with query expansion and deduplication
    
    Args:
        query: Input question
        k: Number of documents to retrieve per query variation
        mu: Dirichlet smoothing parameter for BM25
    """
    searcher.set_qld(mu=mu)
    
    # Preprocess and expand query
    processed_query = preprocess_query(query)
    query_variations = expand_query(processed_query)
    
    # Retrieve documents for each query variation
    all_docs = {}
    
    for q in query_variations:
        hits = searcher.search(q, k)
        
        for hit in hits:
            doc_id = hit.docid
            if doc_id not in all_docs:
                doc = searcher.doc(doc_id)
                raw_json = doc.raw()
                data = json.loads(raw_json)
                contents = data['contents']
                
                # Store both full content and score for reranking
                all_docs[doc_id] = {
                    'content': contents,
                    'score': hit.score,
                    'title': data.get('title', '')
                }
            else:
                # Boost score if document appears for multiple query variations
                all_docs[doc_id]['score'] += hit.score * 0.5
    
    # Sort by score and return top k
    sorted_docs = sorted(all_docs.values(), key=lambda x: x['score'], reverse=True)[:k]
    
    contexts = []
    for doc in sorted_docs:
        # Clean content
        content = doc['content'].replace('\n', ' ')
        # Include title for better context
        if doc['title']:
            content = f"{doc['title']}: {content}"
        contexts.append(content[:500])  # Limit length
    
    return contexts

## Improved Prompt Engineering

In [ ]:
def create_message_improved(query, contexts, prompt_version=1):
    """
    Create optimized prompts for answer extraction
    
    Args:
        query: Input question
        contexts: Retrieved context passages
        prompt_version: Different prompt strategies to try
    """
    # Join contexts with clear separation
    context_text = '\n---\n'.join([f"Document {i+1}: {ctx}" for i, ctx in enumerate(contexts)])
    
    if prompt_version == 1:
        # Direct, concise answer extraction
        system_prompt = (
            "You are a precise question-answering assistant. "
            "Your task is to extract the shortest, most direct answer from the provided documents. "
            "Rules:\n"
            "- Only use information from the documents\n"
            "- Return ONLY the answer, nothing else\n"
            "- Do not add explanations, context, or extra words\n"
            "- If the answer is not in the documents, return 'unknown'"
        )
        
        user_prompt = (
            f"Documents:\n{context_text}\n\n"
            f"Question: {query}\n\n"
            f"Answer (only the answer, nothing else):"
        )
    
    elif prompt_version == 2:
        # Entity-focused extraction
        system_prompt = (
            "Extract the specific entity, name, date, or fact that answers the question. "
            "Return only that entity with no additional text."
        )
        
        user_prompt = (
            f"Context:\n{context_text}\n\n"
            f"Question: {query}\n"
            f"Extract the answer:"
        )
    
    else:  # prompt_version == 3
        # Few-shot style with examples
        system_prompt = (
            "Answer questions with the shortest possible answer from the documents. "
            "Examples:\n"
            "Q: What is the capital of France? A: Paris\n"
            "Q: Who wrote Hamlet? A: William Shakespeare\n"
            "Now answer the following:"
        )
        
        user_prompt = (
            f"Documents: {context_text}\n\n"
            f"Q: {query}\nA:"
        )
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    
    return messages

## Answer Post-processing

In [ ]:
def post_process_answer(answer, question):
    """
    Clean and normalize the generated answer
    """
    # Remove common prefixes
    prefixes_to_remove = [
        'the answer is',
        'answer:',
        'a:',
        'the',
        'according to the document',
        'based on the',
        'from the document',
    ]
    
    answer_lower = answer.lower().strip()
    for prefix in prefixes_to_remove:
        if answer_lower.startswith(prefix):
            answer = answer[len(prefix):].strip()
            answer_lower = answer.lower().strip()
    
    # Remove quotes if present
    answer = answer.strip('"\"''').strip()
    
    # Remove trailing periods if they don't seem to be part of the answer
    if answer.endswith('.') and not answer.endswith('Inc.') and not answer.endswith('Jr.'):
        answer = answer[:-1].strip()
    
    # Handle "unknown" responses
    unknown_responses = ['unknown', 'i dont know', "i don't know", 'not mentioned', 'not found']
    if answer_lower in unknown_responses:
        return "unknown"
    
    # If answer is too long, try to extract the key part
    if len(answer.split()) > 15:
        # Try to find a sentence that contains key question words
        sentences = answer.split('.')
        if len(sentences) > 1:
            answer = sentences[0].strip()
    
    return answer.strip()

## Main Answer Generation Function

In [ ]:
def generate_answer(query, k=10, mu=1000, prompt_version=1, temperature=0.3, max_tokens=100):
    """
    Generate answer for a given query using improved RAG pipeline
    
    Args:
        query: Input question
        k: Number of documents to retrieve
        mu: BM25 smoothing parameter
        prompt_version: Which prompt strategy to use
        temperature: LLM temperature
        max_tokens: Maximum tokens to generate
    """
    # Retrieve context
    contexts = get_context_improved(query, k=k, mu=mu)
    
    if not contexts:
        return "unknown"
    
    # Create prompt
    messages = create_message_improved(query, contexts, prompt_version=prompt_version)
    
    # Generate answer
    try:
        outputs = pipeline(
            messages,
            max_new_tokens=max_tokens,
            eos_token_id=terminators,
            do_sample=True,
            temperature=temperature,
            top_p=0.9,
        )
        
        answer = outputs[0]["generated_text"][-1].get('content', 'unknown')
        
        # Post-process answer
        answer = post_process_answer(answer, query)
        
        return answer
    
    except Exception as e:
        print(f"Error generating answer for '{query}': {e}")
        return "unknown"

## Evaluation Functions

In [ ]:
def normalize_answer(s):
    """Lower text and remove punctuation, articles and extra whitespace."""
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)

    def white_space_fix(text):
        return ' '.join(text.split())

    def remove_punc(text):
        return ''.join(ch for ch in text if ch not in set(string.punctuation))

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))

def f1_score(prediction, ground_truth):
    """Compute token-level F1 score between prediction and a ground truth."""
    pred_tokens = normalize_answer(prediction).split()
    gt_tokens = normalize_answer(ground_truth).split()
    common = Counter(pred_tokens) & Counter(gt_tokens)
    num_same = sum(common.values())

    if len(pred_tokens) == 0 or len(gt_tokens) == 0:
        return int(pred_tokens == gt_tokens)
    if num_same == 0:
        return 0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gt_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    return f1

def metric_max_over_ground_truths(metric_fn, prediction, ground_truths):
    return max(metric_fn(prediction, gt) for gt in ground_truths)

def evaluate(predictions_dict, df_gold):
    """
    Evaluate predictions against ground truth
    """
    f1_sum = 0.0
    count = 0
    
    for idx, row in df_gold.iterrows():
        qid = row['id']
        if qid not in predictions_dict:
            continue
        
        ground_truths = row['answers']
        prediction = predictions_dict[qid]
        
        f1 = metric_max_over_ground_truths(f1_score, prediction, ground_truths)
        f1_sum += f1
        count += 1
    
    return (100.0 * f1_sum / count) if count > 0 else 0.0

## Hyperparameter Tuning on Training Set

In [ ]:
# Test on a sample of training data to find best hyperparameters
sample_size = 100  # Adjust based on compute resources
df_train_sample = df_train.sample(n=min(sample_size, len(df_train)), random_state=42)

# Grid search over hyperparameters
param_grid = {
    'k': [5, 10, 15],
    'mu': [500, 1000, 2000],
    'prompt_version': [1, 2, 3],
    'temperature': [0.1, 0.3, 0.5]
}

best_score = 0
best_params = {}

print("Starting hyperparameter tuning...\n")

# Test a few key combinations (full grid search would be too expensive)
test_configs = [
    {'k': 10, 'mu': 1000, 'prompt_version': 1, 'temperature': 0.3},
    {'k': 10, 'mu': 1000, 'prompt_version': 2, 'temperature': 0.1},
    {'k': 15, 'mu': 500, 'prompt_version': 1, 'temperature': 0.3},
    {'k': 5, 'mu': 2000, 'prompt_version': 1, 'temperature': 0.1},
]

for i, config in enumerate(test_configs):
    print(f"Testing config {i+1}/{len(test_configs)}: {config}")
    
    predictions = {}
    for idx, row in df_train_sample.iterrows():
        question = row['question']
        answer = generate_answer(question, **config)
        predictions[row['id']] = answer
    
    score = evaluate(predictions, df_train_sample)
    print(f"F1 Score: {score:.2f}%\n")
    
    if score > best_score:
        best_score = score
        best_params = config

print(f"\nBest configuration: {best_params}")
print(f"Best F1 Score: {best_score:.2f}%")

## Generate Predictions on Test Set

In [ ]:
# Use best parameters (or manually set based on tuning results)
final_params = {
    'k': 10,
    'mu': 1000,
    'prompt_version': 1,
    'temperature': 0.3,
    'max_tokens': 100
}

print(f"Generating predictions for {len(df_test)} test questions...")
print(f"Using parameters: {final_params}\n")

test_predictions = {}

for idx, row in df_test.iterrows():
    question = row['question']
    qid = row['id']
    
    answer = generate_answer(question, **final_params)
    test_predictions[qid] = answer
    
    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1}/{len(df_test)} questions")
    
    # Print some examples
    if idx < 5:
        print(f"Q: {question}")
        print(f"A: {answer}")
        print("-" * 50)

print("\nPrediction generation complete!")

## Save Predictions

In [ ]:
# Create submission dataframe
submission_df = pd.DataFrame([
    {'id': qid, 'prediction': answer}
    for qid, answer in test_predictions.items()
])

# Sort by id
submission_df = submission_df.sort_values('id').reset_index(drop=True)

# CRITICAL: Format prediction column as JSON list with ensure_ascii=False
# This is required for proper evaluation
submission_df['prediction'] = submission_df['prediction'].apply(
    lambda x: json.dumps([x], ensure_ascii=False)
)

# Verify format before saving
print("Verifying submission format...")
print(f"Total predictions: {len(submission_df)}")
print(f"Expected: {len(df_test)}")
print("\nSample predictions (showing JSON format):")
print(submission_df.head(3))
print("\nFormat check - First prediction:")
print(f"Type: {type(submission_df.iloc[0]['prediction'])}")
print(f"Value: {submission_df.iloc[0]['prediction']}")

# Save to CSV
output_path = "/content/drive/MyDrive/q-a-rag-assignment-3-reichman-uni/improved_predictions.csv"
submission_df.to_csv(output_path, index=False)

print(f"\n✓ Predictions saved to: {output_path}")
print(f"✓ File contains {len(submission_df)} predictions in correct JSON format")

## Optional: Evaluate on Training Set

In [ ]:
# If you want to evaluate on full training set
# WARNING: This will take a long time!

run_full_train_eval = False  # Set to True to run

if run_full_train_eval:
    print("Generating predictions for training set...")
    train_predictions = {}
    
    for idx, row in df_train.iterrows():
        question = row['question']
        answer = generate_answer(question, **final_params)
        train_predictions[row['id']] = answer
        
        if (idx + 1) % 100 == 0:
            print(f"Processed {idx + 1}/{len(df_train)} questions")
    
    final_train_score = evaluate(train_predictions, df_train)
    print(f"\nFinal Training F1 Score: {final_train_score:.2f}%")